[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/08-ai-engineering/10_vector-db.ipynb)

*AI Engineering · Notebook 10 of 10*

# Vector Databases & AI Frameworks

The infrastructure under RAG and agents: distance metrics, IVF, product and scalar quantization, HNSW graph search, post-filtering, upserts and deletes, memory sizing, sharded search, hybrid scoring, metadata filters, vector search in SQL, embedding and semantic caches, model migrations — and how LangChain-style frameworks compose chains, split text, template prompts and retrieve.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** A vector index is a library arranged by meaning: similar books sit on nearby shelves, so you only walk a few aisles instead of the whole building.

**What interviewers really test here:** Metrics and normalisation, HNSW, IVF and quantization trade-offs, filtering, sizing and sharding, pgvector vs dedicated stores, model migrations, and what LangChain or LlamaIndex do underneath.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. What a vector database does *(Basics)*
- [ ] 2. Inside ANN indexes *(Core)*
- [ ] 3. Vector search in practice *(Core)*
- [ ] 4. The same ideas in LangChain and LlamaIndex *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
import numpy as np
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Choosing a vector store

| Option | Best for |
| --- | --- |
| pgvector | already on Postgres, up to tens of millions of vectors, relational filters and joins |
| Qdrant / Weaviate / Milvus | large scale, rich filtering, hybrid search, self-hosted or cloud |
| Pinecone and other managed services | no operations, pay per use |
| Chroma | prototypes and notebooks |
| FAISS | a fast in-process library, no server, no metadata store |
| DuckDB / LanceDB / OpenSearch | analytics, embedded files, or adding vectors to existing search |

### HNSW parameters

- `M` — links per node: more links, better recall, more memory.
- `ef_construction` — effort while building: slower builds, better graphs.
- `ef_search` — candidates explored per query: the main recall-vs-latency dial at query time.

---
# Part 2: Lessons, step by step

## Lesson 1: What a vector database does

*Level: Basics*

A vector database stores embeddings with an id and metadata, and answers "which stored vectors are closest to this one?" fast — with filters, updates and deletes, at millions to billions of vectors. Exact search compares the query with every vector (fine up to ~100k); beyond that you use an **approximate nearest neighbour (ANN)** index.




- **Metric**: cosine, dot product or Euclidean. With normalised vectors they rank the same; use what your embedding model was trained for.
- **Filters**: metadata conditions (tenant, language, date). Filter *during* search; filtering after a top-k search can leave you with too few results.
- **Writes**: upserts and deletes go to segments with tombstones that compaction cleans up.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def normalise_demo():
    docs = np.array([[0.9, 0.1], [3.0, 1.0]])
    q = np.array([1.0, 0.2])
    print("dot   :", docs @ q, "-> the long vector wins")
    unit = docs / np.linalg.norm(docs, axis=1, keepdims=True)
    print("cosine:", np.round(unit @ (q / np.linalg.norm(q)), 3), "-> direction wins")


normalise_demo()

dot   : [0.92 3.2 ] -> the long vector wins
cosine: [0.996 0.992] -> direction wins


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q14](#q14), [Q11](#q11). Try one now, or carry on and do them in Part 3.

## Lesson 2: Inside ANN indexes

*Level: Core*

| Index | How it works | Trade-off |
| --- | --- | --- |
| **HNSW** | a layered graph; search walks greedily towards the query | fast, high recall, memory-hungry |
| **IVF** | cluster vectors; scan only the nearest `nprobe` clusters | tune nprobe for recall vs speed |
| **PQ** | split vectors into parts, store each as a codebook index | 32–64× smaller, some accuracy lost |
| **Scalar (int8)** | store each number in one byte | 4× smaller, tiny error |



Size the index before choosing hardware: 10 million 1536-dimension float32 vectors are ~61 GB before the graph. Big indexes are **sharded**: each shard returns its top k and a coordinator merges them.

**✏️ Your turn:** you now have what you need for [Q8](#q8), [Q10](#q10), [Q20](#q20), [Q9](#q9), [Q3](#q3), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 3: Vector search in practice

*Level: Core*

Start with what you already run. **pgvector** adds vectors to PostgreSQL, so tenant filters, joins and transactions just work; dedicated stores (Qdrant, Weaviate, Milvus, Pinecone) add scale and features; **Chroma** and **FAISS** are great for prototypes and in-process search.



```python
# pgvector
SELECT id, body, 1 - (embedding <=> :q) AS similarity
FROM chunks WHERE tenant = :tenant
ORDER BY embedding <=> :q LIMIT 5;

# Qdrant
from qdrant_client import QdrantClient, models
client = QdrantClient(":memory:")
client.create_collection("docs", vectors_config=models.VectorParams(size=384, distance=models.Distance.COSINE))
client.upsert("docs", points=[models.PointStruct(id=1, vector=v, payload={"lang": "en"})])
client.query_points("docs", query=q, limit=5,
    query_filter=models.Filter(must=[models.FieldCondition(key="lang", match=models.MatchValue(value="en"))]))

# Chroma
import chromadb
col = chromadb.Client().create_collection("docs")
col.add(ids=["c1"], documents=["refund policy"], metadatas=[{"lang": "en"}])
col.query(query_texts=["how do refunds work"], n_results=3, where={"lang": "en"})
```



Combine keyword and vector scores (**hybrid search**) for names, codes and rare words; cache embeddings by model + text hash; and plan model migrations as a backfill into a new index, then a switch.

**✏️ Your turn:** you now have what you need for [Q15](#q15), [Q13](#q13), [Q4](#q4), [Q16](#q16), [Q19](#q19), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 4: The same ideas in LangChain and LlamaIndex

*Level: Advanced*

Frameworks package these pieces — loaders, splitters, embeddings, vector stores, retrievers, prompts, models and parsers — behind common interfaces, so you can swap providers. Everything in this course maps onto them:



```python
# LangChain (LCEL): steps composed with |
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_anthropic import ChatAnthropic

chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_documents(docs)
store = Chroma.from_documents(chunks, embeddings)                  # or PGVector, Qdrant…
retriever = store.as_retriever(search_type="similarity_score_threshold",
                               search_kwargs={"k": 4, "score_threshold": 0.75})
prompt = ChatPromptTemplate.from_template("Use only this context:\n{context}\n\nQuestion: {question}")
chain = ({"context": retriever, "question": RunnablePassthrough()}
         | prompt | ChatAnthropic(model="claude-sonnet-5-5") | StrOutputParser())
chain.invoke("How long do refunds take?")

# LlamaIndex: index → query engine
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
index = VectorStoreIndex.from_documents(SimpleDirectoryReader("docs").load_data())
index.as_query_engine(similarity_top_k=4).query("How long do refunds take?")
```



**When to use one:** frameworks speed up prototypes and give many integrations; for production paths, many teams keep the core loop in plain code (as this course does) and use framework pieces where they help. Either way, interviewers expect you to explain what happens inside — which is what the questions here practise.

**✏️ Your turn:** you now have what you need for [Q17](#q17), [Q18](#q18), [Q5](#q5), [Q7](#q7). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Cosine, Dot or Euclidean?

🟢 Warm-up · *Metrics rank differently*

Rank the documents for the query by cosine similarity, dot product (both higher is better) and Euclidean distance (lower is better). Return the three rankings, then the three rankings again after normalising every vector to length 1 — they agree once vectors are normalised, which is why most embedding models ship normalised vectors.

**Examples**

- `docs = {'d1': [0.9, 0.1, 0.0], 'd2': [3.0, 1.0, 0.2], 'd3': [0.2, 0.9, 0.1], 'd4': [0.5, 0.5, 0.5], 'd5': [0.05, 0.02, 0.0]}, query = [1.0, 0.2, 0.0]` → `[['d1', 'd2', 'd5', 'd4', 'd3'], ['d2', 'd1', 'd4', 'd3', 'd5'], ['d1', 'd4', 'd5', 'd3', 'd2'], ['d1', 'd2', 'd5', 'd4', 'd3'], ['d1', 'd2', 'd5', 'd4', 'd3'], ['d1', 'd2', 'd5', 'd4', 'd3']]`

In [ ]:
def metric_rankings(docs, query):
    """
    Rank the documents for the query by cosine similarity, dot product (both
    higher is better) and Euclidean distance (lower is better). Return the three
    rankings, then the three rankings again after normalising every vector to
    length 1 — they agree once vectors are normalised, which is why most
    embedding models ship normalised vectors.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(metric_rankings, [
    (({'d1': [0.9, 0.1, 0.0], 'd2': [3.0, 1.0, 0.2], 'd3': [0.2, 0.9, 0.1], 'd4': [0.5, 0.5, 0.5], 'd5': [0.05, 0.02, 0.0]}, [1.0, 0.2, 0.0]), [['d1', 'd2', 'd5', 'd4', 'd3'], ['d2', 'd1', 'd4', 'd3', 'd5'], ['d1', 'd4', 'd5', 'd3', 'd2'], ['d1', 'd2', 'd5', 'd4', 'd3'], ['d1', 'd2', 'd5', 'd4', 'd3'], ['d1', 'd2', 'd5', 'd4', 'd3']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute each metric with loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Matrix operations in NumPy.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute each metric with loops.

```python
def metric_rankings(docs, query):
    def rank(vectors, q):
        cos, dot, l2 = {}, {}, {}
        for k, v in vectors.items():
            d = sum(a * b for a, b in zip(v, q))
            nv = math.sqrt(sum(a * a for a in v))
            nq = math.sqrt(sum(a * a for a in q))
            cos[k] = d / (nv * nq)
            dot[k] = d
            l2[k] = math.sqrt(sum((a - b) ** 2 for a, b in zip(v, q)))
        return [sorted(cos, key=lambda k: -cos[k]), sorted(dot, key=lambda k: -dot[k]), sorted(l2, key=lambda k: l2[k])]
    unit = lambda v: [x / math.sqrt(sum(y * y for y in v)) for x in v]
    return rank(docs, query) + rank({k: unit(v) for k, v in docs.items()}, unit(query))
```

**Cost:** O(n · d) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Matrix operations in NumPy.

```python
def metric_rankings(docs, query):
    names = list(docs)
    def rank(M, q):
        dot = M @ q
        cos = dot / (np.linalg.norm(M, axis=1) * np.linalg.norm(q))
        l2 = np.linalg.norm(M - q, axis=1)
        order = lambda s: [names[i] for i in np.argsort(s, kind="stable")]
        return [order(-cos), order(-dot), order(l2)]
    M, q = np.array([docs[n] for n in names], dtype=float), np.array(query, dtype=float)
    return rank(M, q) + rank(M / np.linalg.norm(M, axis=1, keepdims=True), q / np.linalg.norm(q))
```

**Cost:** O(n · d) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute each metric with loops. That costs O(n · d) time and O(n) space."

"I can do better: Matrix operations in NumPy. That's O(n · d) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `docs = {'a': [1.0, 0.0], 'b': [3.0, 3.0]}, query = [1.0, 0.1]`

1. rank() returns [['a', 'b'], ['b', 'a'], ['a', 'b']]
2. rank() returns [['a', 'b'], ['a', 'b'], ['a', 'b']]
3. metric\_rankings() returns [['a', 'b'], ['b', 'a'], ['a', 'b'], ...

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/cosine-dot-or-euclidean)

<a id="q2"></a>
### Q2. Post-Filtering Loses Results

🟢 Warm-up · *Filter before or after the search?*

Documents have a similarity score to the query and a `lang`. Post-filtering takes the top `k` by score and then keeps the matching language; pre-filtering keeps the language first, then takes the top k. Return both result lists — post-filtering can return fewer than k even when enough matching documents exist.

**Examples**

- `docs = {'d1': (0.92, 'en'), 'd2': (0.9, 'en'), 'd3': (0.88, 'hi'), 'd4': (0.85, 'en'), 'd5': (0.8, 'hi'), 'd6': (0.71, 'hi'), 'd7': (0.65, 'en')}, lang = 'hi', k = 3` → `(['d3'], ['d3', 'd5', 'd6'])`

In [ ]:
def pre_vs_post(docs, lang, k):
    """
    Documents have a similarity score to the query and a lang. Post-filtering
    takes the top k by score and then keeps the matching language; pre-filtering
    keeps the language first, then takes the top k. Return both result lists —
    post-filtering can return fewer than k even when enough matching documents
    exist.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pre_vs_post, [
    (({'d1': (0.92, 'en'), 'd2': (0.9, 'en'), 'd3': (0.88, 'hi'), 'd4': (0.85, 'en'), 'd5': (0.8, 'hi'), 'd6': (0.71, 'hi'), 'd7': (0.65, 'en')}, 'hi', 3), (['d3'], ['d3', 'd5', 'd6'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort, slice, filter by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Comprehensions around sorted().

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sort, slice, filter by hand.

```python
def pre_vs_post(docs, lang, k):
    ranked = sorted(docs, key=lambda d: -docs[d][0])
    post = []
    for d in ranked[:k]:
        if docs[d][1] == lang:
            post.append(d)
    pre = []
    for d in ranked:
        if docs[d][1] == lang and len(pre) < k:
            pre.append(d)
    return (post, pre)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Comprehensions around sorted().

```python
def pre_vs_post(docs, lang, k):
    ranked = sorted(docs, key=lambda d: -docs[d][0])
    return ([d for d in ranked[:k] if docs[d][1] == lang], [d for d in ranked if docs[d][1] == lang][:k])
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort, slice, filter by hand. That costs O(n log n) time and O(n) space."

"I can do better: Comprehensions around sorted(). That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/post-filtering-loses-results)

<a id="q3"></a>
### Q3. How Much Memory Will the Index Need?

🟢 Warm-up · *Size before you buy*

For `n` vectors of dimension `d`, return GB (rounded to 1) for: float32 vectors, plus an HNSW graph with `M` links per node in both directions (2·M × 4-byte ids on the base layer), and the same with int8 vectors, and PQ with `m` bytes per vector (no graph). 1 GB = 10⁹ bytes.

**Examples**

- `n = 10000000, d = 1536, M = 16, m = 96` → `{'float32+hnsw': 62.7, 'int8+hnsw': 16.6, 'pq': 1.0}`
- `n = 500000, d = 384, M = 16, m = 48` → `{'float32+hnsw': 0.8, 'int8+hnsw': 0.3, 'pq': 0.0}`

In [ ]:
def index_memory(n, d, M, m):
    """
    For n vectors of dimension d, return GB (rounded to 1) for: float32 vectors,
    plus an HNSW graph with M links per node in both directions (2·M × 4-byte
    ids on the base layer), and the same with int8 vectors, and PQ with m bytes
    per vector (no graph). 1 GB = 10⁹ bytes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(index_memory, [
    ((10000000, 1536, 16, 96), {'float32+hnsw': 62.7, 'int8+hnsw': 16.6, 'pq': 1.0}),
    ((500000, 384, 16, 48), {'float32+hnsw': 0.8, 'int8+hnsw': 0.3, 'pq': 0.0}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Arithmetic step by step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, compact.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Arithmetic step by step.

```python
def index_memory(n, d, M, m):
    graph = n * 2 * M * 4
    f32 = n * d * 4 + graph
    i8 = n * d + graph
    pq = n * m
    return {"float32+hnsw": round(f32 / 1e9, 1), "int8+hnsw": round(i8 / 1e9, 1), "pq": round(pq / 1e9, 1)}
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, compact.

```python
def index_memory(n, d, M, m):
    g = n * 8 * M
    return {k: round(v / 1e9, 1) for k, v in {"float32+hnsw": n * d * 4 + g, "int8+hnsw": n * d + g, "pq": n * m}.items()}
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Arithmetic step by step. That costs O(1) time and O(1) space."

"I can do better: Same, compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/how-much-memory-will-the-index-need)

<a id="q4"></a>
### Q4. Cache Embeddings

🟢 Warm-up · *Never pay twice for the same text*

Embedding calls cost money and time. Cache by `model + sha256(text)`. For the stream of `(model, text)` requests, return `(API calls made, cache hits, % saved rounded to 1)`. Changing the model must miss the cache — vectors from different models aren't comparable.

**Examples**

- `requests = [('e5-small', 'refund policy'), ('e5-small', 'shipping times'), ('e5-small', 'refund policy'), ('bge-m3', 'refund policy'), ('e5-small', 'refund policy'), ('bge-m3', 'refund policy'), ('e5-small', 'Refund policy')]` → `(4, 3, 42.9)`

In [ ]:
import hashlib

def embedding_cache(requests):
    """
    Embedding calls cost money and time. Cache by model + sha256(text). For the
    stream of (model, text) requests, return (API calls made, cache hits, %
    saved rounded to 1). Changing the model must miss the cache — vectors from
    different models aren't comparable.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(embedding_cache, [
    (([('e5-small', 'refund policy'), ('e5-small', 'shipping times'), ('e5-small', 'refund policy'), ('bge-m3', 'refund policy'), ('e5-small', 'refund policy'), ('bge-m3', 'refund policy'), ('e5-small', 'Refund policy')],), (4, 3, 42.9)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list of seen keys.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A set of hashed keys.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A list of seen keys.

```python
def embedding_cache(requests):
    seen, calls, hits = [], 0, 0
    for model, text in requests:
        key = model + ":" + hashlib.sha256(text.encode()).hexdigest()
        if key in seen:
            hits += 1
        else:
            seen.append(key)
            calls += 1
    return (calls, hits, round(hits / len(requests) * 100, 1))
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A set of hashed keys.

```python
def embedding_cache(requests):
    keys = [f"{m}:{hashlib.sha256(t.encode()).hexdigest()}" for m, t in requests]
    calls = len(set(keys))
    return (calls, len(keys) - calls, round(100 * (len(keys) - calls) / len(keys), 1))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list of seen keys. That costs O(n²) time and O(n) space."

"I can do better: A set of hashed keys. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/cache-embeddings)

<a id="q5"></a>
### Q5. Fill a Prompt Template Safely

🟢 Warm-up · *PromptTemplate variables*

A template has `{variables}` (`{{` and `}}` are literal braces). For each input dict, return the filled prompt, or an error listing missing variables (sorted) — and separately list unused inputs (sorted). Catching a missing variable at build time beats sending `'{context}'` to the model.

**Examples**

- `template = 'Use only this context:\n{context}\n\nQuestion: {question}\nAnswer as JSON like {{"answer": ...}}.', inputs = [{'context': 'Refunds take 5 days.', 'question': 'How long do refunds take?'}, {'question': 'Hi', 'user': 'asha'}]` → `[('Use only this context:\nRefunds take 5 days.\n\nQuestion: How long do refunds take?\nAnswer as JSON like {"answer": ...}.', []), ('error: missing context', ['user'])]`

In [ ]:
def fill_template(template, inputs):
    """
    A template has {variables} ({{ and }} are literal braces). For each input
    dict, return the filled prompt, or an error listing missing variables
    (sorted) — and separately list unused inputs (sorted). Catching a missing
    variable at build time beats sending '{context}' to the model.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fill_template, [
    (('Use only this context:\n{context}\n\nQuestion: {question}\nAnswer as JSON like {{"answer": ...}}.', [{'context': 'Refunds take 5 days.', 'question': 'How long do refunds take?'}, {'question': 'Hi', 'user': 'asha'}]), [('Use only this context:\nRefunds take 5 days.\n\nQuestion: How long do refunds take?\nAnswer as JSON like {"answer": ...}.', []), ('error: missing context', ['user'])]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Scan for braces by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`string.Formatter().parse` and `str.format`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Scan for braces by hand.

```python
def fill_template(template, inputs):
    names, i = [], 0
    while i < len(template):
        if template.startswith("{{", i) or template.startswith("}}", i):
            i += 2
        elif template[i] == "{":
            j = template.index("}", i)
            if template[i + 1:j] not in names:
                names.append(template[i + 1:j])
            i = j + 1
        else:
            i += 1
    out = []
    for vals in inputs:
        missing = sorted(n for n in names if n not in vals)
        unused = sorted(k for k in vals if k not in names)
        filled = None if missing else template.format(**{n: vals[n] for n in names})
        out.append((filled if filled else "error: missing " + ", ".join(missing), unused))
    return out
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `string.Formatter().parse` and `str.format`.

```python
def fill_template(template, inputs):
    import string
    names = {f for _, f, _, _ in string.Formatter().parse(template) if f}
    out = []
    for vals in inputs:
        missing = sorted(names - vals.keys())
        out.append((f"error: missing {', '.join(missing)}" if missing else template.format(**vals), sorted(vals.keys() - names)))
    return out
```

**Cost:** O(n · L) time, O(L) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Scan for braces by hand. That costs O(n · L) time and O(L) space."

"I can do better: `string.Formatter().parse` and `str.format`. That's O(n · L) time and O(L) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/fill-a-prompt-template-safely)

<a id="q6"></a>
### Q6. Choose a Vector Store

🟢 Warm-up · *Match the tool to the job*

Recommend a store for each situation (a dict of facts) using these rules in order: already on Postgres and under 10 million vectors → `'pgvector'`; a prototype or notebook → `'Chroma'`; in-process search only, no server → `'FAISS'`; need no ops at all → `'managed (e.g. Pinecone)'`; otherwise → `'Qdrant or Weaviate'`.

**Examples**

- `cases = [{'postgres': True, 'vectors': 2000000}, {'prototype': True}, {'postgres': True, 'vectors': 80000000, 'no_ops': True}, {'in_process': True, 'vectors': 5000000}, {'vectors': 300000000, 'filters': True}]` → `['pgvector', 'Chroma', 'managed (e.g. Pinecone)', 'FAISS', 'Qdrant or Weaviate']`

In [ ]:
def choose_store(cases):
    """
    Recommend a store for each situation (a dict of facts) using these rules in
    order: already on Postgres and under 10 million vectors → 'pgvector'; a
    prototype or notebook → 'Chroma'; in-process search only, no server →
    'FAISS'; need no ops at all → 'managed (e.g. Pinecone)'; otherwise → 'Qdrant
    or Weaviate'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(choose_store, [
    (([{'postgres': True, 'vectors': 2000000}, {'prototype': True}, {'postgres': True, 'vectors': 80000000, 'no_ops': True}, {'in_process': True, 'vectors': 5000000}, {'vectors': 300000000, 'filters': True}],), ['pgvector', 'Chroma', 'managed (e.g. Pinecone)', 'FAISS', 'Qdrant or Weaviate']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

An if/elif chain.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A rule table checked in order.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** An if/elif chain.

```python
def choose_store(cases):
    out = []
    for c in cases:
        if c.get("postgres") and c.get("vectors", 0) < 10_000_000:
            out.append("pgvector")
        elif c.get("prototype"):
            out.append("Chroma")
        elif c.get("in_process"):
            out.append("FAISS")
        elif c.get("no_ops"):
            out.append("managed (e.g. Pinecone)")
        else:
            out.append("Qdrant or Weaviate")
    return out
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A rule table checked in order.

```python
def choose_store(cases):
    rules = [(lambda c: c.get("postgres") and c.get("vectors", 0) < 10_000_000, "pgvector"), (lambda c: c.get("prototype"), "Chroma"),
             (lambda c: c.get("in_process"), "FAISS"), (lambda c: c.get("no_ops"), "managed (e.g. Pinecone)")]
    return [next((name for test, name in rules if test(c)), "Qdrant or Weaviate") for c in cases]
```

**Cost:** O(n · r) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: An if/elif chain. That costs O(n) time and O(1) space."

"I can do better: A rule table checked in order. That's O(n · r) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/choose-a-vector-store)

<a id="q7"></a>
### Q7. A Retriever With a Score Threshold

🟢 Warm-up · *search\_type='similarity\_score\_threshold'*

Frameworks wrap a vector store as a **retriever**: return at most `k` documents, but only those with similarity ≥ `threshold` — so an off-topic question retrieves nothing instead of junk. Given each question's similarity scores, return the retrieved ids per question (highest score first, ties by id).

**Examples**

- `score_sets = [{'d1': 0.82, 'd2': 0.79, 'd3': 0.41, 'd4': 0.77}, {'d1': 0.31, 'd2': 0.28, 'd3': 0.35}, {'d5': 0.91, 'd6': 0.9}], threshold = 0.75, k = 2` → `[['d1', 'd2'], [], ['d5', 'd6']]`

In [ ]:
def threshold_retriever(score_sets, threshold, k):
    """
    Frameworks wrap a vector store as a retriever: return at most k documents,
    but only those with similarity ≥ threshold — so an off-topic question
    retrieves nothing instead of junk. Given each question's similarity scores,
    return the retrieved ids per question (highest score first, ties by id).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(threshold_retriever, [
    (([{'d1': 0.82, 'd2': 0.79, 'd3': 0.41, 'd4': 0.77}, {'d1': 0.31, 'd2': 0.28, 'd3': 0.35}, {'d5': 0.91, 'd6': 0.9}], 0.75, 2), [['d1', 'd2'], [], ['d5', 'd6']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort, then filter in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension over sorted items.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sort, then filter in a loop.

```python
def threshold_retriever(score_sets, threshold, k):
    out = []
    for scores in score_sets:
        ranked = sorted(scores, key=lambda d: (-scores[d], d))
        got = []
        for d in ranked:
            if scores[d] >= threshold and len(got) < k:
                got.append(d)
        out.append(got)
    return out
```

**Cost:** O(q · n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension over sorted items.

```python
def threshold_retriever(score_sets, threshold, k):
    return [[d for d in sorted(s, key=lambda d: (-s[d], d)) if s[d] >= threshold][:k] for s in score_sets]
```

**Cost:** O(q · n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort, then filter in a loop. That costs O(q · n log n) time and O(n) space."

"I can do better: A comprehension over sorted items. That's O(q · n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/a-retriever-with-a-score-threshold)

<a id="q8"></a>
### Q8. Search an IVF Index

🟡 Interview standard · *Inverted file lists*

An IVF index assigns every vector to its nearest centroid (Euclidean). A query only scans the lists of its `nprobe` nearest centroids. For each nprobe, return `(vectors scanned, top-k ids found, recall vs exact top-k)`.

**Examples**

- `points = {'p1': [0.1, 0.2], 'p2': [0.2, 0.1], 'p3': [0.9, 0.8], 'p4': [0.8, 0.95], 'p5': [0.15, 0.9], 'p6': [0.25, 0.8], 'p7': [0.85, 0.15], 'p8': [0.5, 0.45], 'p9': [0.55, 0.6], 'p10': [0.95, 0.3]}, centroids = [[0.15, 0.15], [0.85, 0.85], [0.2, 0.85], [0.85, 0.2]], query = [0.45, 0.5], k = 3, nprobes = [1, 2, 4]` → `[(2, ['p6', 'p5'], 0.33), (4, ['p6', 'p1', 'p2'], 0.33), (10, ['p8', 'p9', 'p6'], 1.0)]`

In [ ]:
from collections import defaultdict

def ivf_search(points, centroids, query, k, nprobes):
    """
    An IVF index assigns every vector to its nearest centroid (Euclidean). A
    query only scans the lists of its nprobe nearest centroids. For each nprobe,
    return (vectors scanned, top-k ids found, recall vs exact top-k).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(ivf_search, [
    (({'p1': [0.1, 0.2], 'p2': [0.2, 0.1], 'p3': [0.9, 0.8], 'p4': [0.8, 0.95], 'p5': [0.15, 0.9], 'p6': [0.25, 0.8], 'p7': [0.85, 0.15], 'p8': [0.5, 0.45], 'p9': [0.55, 0.6], 'p10': [0.95, 0.3]}, [[0.15, 0.15], [0.85, 0.85], [0.2, 0.85], [0.85, 0.2]], [0.45, 0.5], 3, [1, 2, 4]), [(2, ['p6', 'p5'], 0.33), (4, ['p6', 'p1', 'p2'], 0.33), (10, ['p8', 'p9', 'p6'], 1.0)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loops for assignment and scanning.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

NumPy distances and argsort.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loops for assignment and scanning.

```python
def ivf_search(points, centroids, query, k, nprobes):
    dist = lambda a, b: math.sqrt(sum((x - y) ** 2 for x, y in zip(a, b)))
    lists = defaultdict(list)
    for pid, v in points.items():
        best = min(range(len(centroids)), key=lambda c: dist(v, centroids[c]))
        lists[best].append(pid)
    exact = sorted(points, key=lambda p: (dist(points[p], query), p))[:k]
    order = sorted(range(len(centroids)), key=lambda c: dist(query, centroids[c]))
    out = []
    for nprobe in nprobes:
        cand = [p for c in order[:nprobe] for p in lists[c]]
        found = sorted(cand, key=lambda p: (dist(points[p], query), p))[:k]
        out.append((len(cand), found, round(len(set(found) & set(exact)) / k, 2)))
    return out
```

**Cost:** O(n · c + p · n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** NumPy distances and argsort.

```python
def ivf_search(points, centroids, query, k, nprobes):
    ids = list(points)
    X, C, q = np.array([points[i] for i in ids]), np.array(centroids), np.array(query)
    assign = np.argmin(((X[:, None, :] - C[None]) ** 2).sum(-1), axis=1)
    d = np.linalg.norm(X - q, axis=1)
    rank = lambda idx: [ids[i] for i in sorted(idx, key=lambda i: (d[i], ids[i]))][:k]
    exact = set(rank(range(len(ids))))
    probe_order = np.argsort(np.linalg.norm(C - q, axis=1), kind="stable")
    out = []
    for n in nprobes:
        cand = [i for i in range(len(ids)) if assign[i] in set(probe_order[:n].tolist())]
        found = rank(cand)
        out.append((len(cand), found, round(len(set(found) & exact) / k, 2)))
    return out
```

**Cost:** O(n · c + p · n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loops for assignment and scanning. That costs O(n · c + p · n) time and O(n) space."

"I can do better: NumPy distances and argsort. That's O(n · c + p · n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/search-an-ivf-index)

<a id="q9"></a>
### Q9. Scalar Quantization to int8

🟡 Interview standard · *4× smaller, tiny error*

Quantize each dimension to an integer 0–255 using that dimension's min and max over all vectors: `code = round((x − min) / (max − min) × 255)`. Return the codes, the largest absolute error after decoding (rounded to 4), and the memory ratio float32 ÷ int8.

**Examples**

- `vectors = [[0.12, -0.5, 0.33], [0.9, 0.1, -0.2], [-0.3, 0.45, 0.05], [0.5, -0.05, 0.8]]` → `([[89, 0, 135], [255, 161, 0], [0, 255, 64], [170, 121, 255]], 0.0012, 4.0)`

In [ ]:
def int8_quantize(vectors):
    """
    Quantize each dimension to an integer 0–255 using that dimension's min and
    max over all vectors: code = round((x − min) / (max − min) × 255). Return
    the codes, the largest absolute error after decoding (rounded to 4), and the
    memory ratio float32 ÷ int8.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(int8_quantize, [
    (([[0.12, -0.5, 0.33], [0.9, 0.1, -0.2], [-0.3, 0.45, 0.05], [0.5, -0.05, 0.8]],), ([[89, 0, 135], [255, 161, 0], [0, 255, 64], [170, 121, 255]], 0.0012, 4.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loops over dimensions and values.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Vectorised min, max and rounding.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loops over dimensions and values.

```python
def int8_quantize(vectors):
    d = len(vectors[0])
    lo = [min(v[j] for v in vectors) for j in range(d)]
    hi = [max(v[j] for v in vectors) for j in range(d)]
    codes, worst = [], 0.0
    for v in vectors:
        row = []
        for j in range(d):
            c = round((v[j] - lo[j]) / (hi[j] - lo[j]) * 255)
            row.append(c)
            back = lo[j] + c / 255 * (hi[j] - lo[j])
            worst = max(worst, abs(back - v[j]))
        codes.append(row)
    return (codes, round(worst, 4), 4.0)
```

**Cost:** O(n · d) time, O(n · d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Vectorised min, max and rounding.

```python
def int8_quantize(vectors):
    X = np.array(vectors)
    lo, hi = X.min(0), X.max(0)
    codes = np.round((X - lo) / (hi - lo) * 255)
    back = lo + codes / 255 * (hi - lo)
    return (codes.astype(int).tolist(), round(float(np.abs(back - X).max()), 4), 4.0)
```

**Cost:** O(n · d) time, O(n · d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loops over dimensions and values. That costs O(n · d) time and O(n · d) space."

"I can do better: Vectorised min, max and rounding. That's O(n · d) time and O(n · d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/scalar-quantization-to-int8)

<a id="q10"></a>
### Q10. Greedy Search on an HNSW Graph

🟡 Interview standard · *Walk towards the query*

HNSW stores vectors as a graph where each node links to near neighbours. Search starts at an entry point and repeatedly moves to the neighbour closest to the query, stopping when no neighbour is closer. Return `(path taken, node found, true nearest node, distance computations made)` for each query.

**Examples**

- `graph = {'a': ['b', 'c'], 'b': ['a', 'd', 'e'], 'c': ['a', 'f'], 'd': ['b', 'g'], 'e': ['b', 'g', 'h'], 'f': ['c', 'h'], 'g': ['d', 'e', 'h'], 'h': ['e', 'f', 'g']}, coords = {'a': [0.0, 0.0], 'b': [1.0, 0.5], 'c': [0.5, 1.5], 'd': [2.0, 0.2], 'e': [2.0, 1.2], 'f': [1.0, 2.5], 'g': [3.0, 0.8], 'h': [2.8, 2.0]}, entry = 'a', queries = [[2.9, 1.9], [2.1, 0.3], [0.9, 2.3]]` → `[(['a', 'b', 'e', 'h'], 'h', 'h', 12), (['a', 'b', 'd'], 'd', 'd', 8), (['a', 'c', 'f'], 'f', 'f', 7)]`

In [ ]:
def hnsw_greedy(graph, coords, entry, queries):
    """
    HNSW stores vectors as a graph where each node links to near neighbours.
    Search starts at an entry point and repeatedly moves to the neighbour
    closest to the query, stopping when no neighbour is closer. Return (path
    taken, node found, true nearest node, distance computations made) for each
    query.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(hnsw_greedy, [
    (({'a': ['b', 'c'], 'b': ['a', 'd', 'e'], 'c': ['a', 'f'], 'd': ['b', 'g'], 'e': ['b', 'g', 'h'], 'f': ['c', 'h'], 'g': ['d', 'e', 'h'], 'h': ['e', 'f', 'g']}, {'a': [0.0, 0.0], 'b': [1.0, 0.5], 'c': [0.5, 1.5], 'd': [2.0, 0.2], 'e': [2.0, 1.2], 'f': [1.0, 2.5], 'g': [3.0, 0.8], 'h': [2.8, 2.0]}, 'a', [[2.9, 1.9], [2.1, 0.3], [0.9, 2.3]]), [(['a', 'b', 'e', 'h'], 'h', 'h', 12), (['a', 'b', 'd'], 'd', 'd', 8), (['a', 'c', 'f'], 'f', 'f', 7)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare every neighbour with explicit loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

min() over neighbours with a cached distance function.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compare every neighbour with explicit loops.

```python
def hnsw_greedy(graph, coords, entry, queries):
    out = []
    for q in queries:
        dist = lambda n: math.dist(coords[n], q)
        cur, path, comps = entry, [entry], 1
        while True:
            best = cur
            for nb in graph[cur]:
                comps += 1
                if dist(nb) < dist(best):
                    best = nb
            if best == cur:
                break
            cur = best
            path.append(cur)
        out.append((path, cur, min(coords, key=dist), comps))
    return out
```

**Cost:** O(hops · degree) time, O(hops) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** min() over neighbours with a cached distance function.

```python
def hnsw_greedy(graph, coords, entry, queries):
    out = []
    for q in queries:
        d = {n: math.dist(c, q) for n, c in coords.items()}
        cur, path, comps = entry, [entry], 1
        while (nxt := min(graph[cur], key=d.get)) and d[nxt] < d[cur]:
            comps += len(graph[cur])
            cur = nxt
            path.append(cur)
        comps += len(graph[cur])
        out.append((path, cur, min(d, key=d.get), comps))
    return out
```

**Cost:** O(hops · degree) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare every neighbour with explicit loops. That costs O(hops · degree) time and O(hops) space."

"I can do better: min() over neighbours with a cached distance function. That's O(hops · degree) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/greedy-search-on-an-hnsw-graph)

<a id="q11"></a>
### Q11. Upserts, Deletes and Compaction

🟡 Interview standard · *Segments and tombstones*

Many vector stores append writes to segments: an upsert adds a new version of an id and hides older ones, a delete writes a tombstone. Process operations `('upsert', id)` and `('delete', id)` and return `(live ids sorted, stored entries before compaction, stored entries after compaction)`. Compaction keeps only live versions.

**Examples**

- `ops = [('upsert', 'a'), ('upsert', 'b'), ('upsert', 'a'), ('delete', 'b'), ('upsert', 'c'), ('upsert', 'a'), ('delete', 'zz'), ('upsert', 'b')]` → `(['a', 'b', 'c'], 8, 3)`

In [ ]:
def segments(ops):
    """
    Many vector stores append writes to segments: an upsert adds a new version
    of an id and hides older ones, a delete writes a tombstone. Process
    operations ('upsert', id) and ('delete', id) and return (live ids sorted,
    stored entries before compaction, stored entries after compaction).
    Compaction keeps only live versions.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(segments, [
    (([('upsert', 'a'), ('upsert', 'b'), ('upsert', 'a'), ('delete', 'b'), ('upsert', 'c'), ('upsert', 'a'), ('delete', 'zz'), ('upsert', 'b')],), (['a', 'b', 'c'], 8, 3)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Append to a log and replay it.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A dict of latest state plus a counter.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Append to a log and replay it.

```python
def segments(ops):
    log = []
    for op, i in ops:
        log.append((op, i))
    live = {}
    for op, i in log:
        if op == "upsert":
            live[i] = True
        else:
            live.pop(i, None)
    return (sorted(live), len(log), len(live))
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A dict of latest state plus a counter.

```python
def segments(ops):
    state = {}
    for op, i in ops:
        state[i] = op == "upsert"
    live = sorted(i for i, alive in state.items() if alive)
    return (live, len(ops), len(live))
```

**Cost:** O(n) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Append to a log and replay it. That costs O(n) time and O(n) space."

"I can do better: A dict of latest state plus a counter. That's O(n) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/upserts-deletes-and-compaction)

<a id="q12"></a>
### Q12. Search Across Shards

🟡 Interview standard · *Scatter, then merge*

A big index is split across shards. Each shard returns its own top `k` (score, id); the coordinator merges them into the global top k. Return the global top-k ids and whether it equals searching all scores at once (it always does when every shard returns k).

**Examples**

- `shards = [{'a': 0.91, 'b': 0.55, 'c': 0.77}, {'d': 0.95, 'e': 0.4, 'f': 0.88}, {'g': 0.6, 'h': 0.93, 'i': 0.79}], k = 4` → `(['d', 'h', 'a', 'f'], True)`

In [ ]:
import heapq

def shard_search(shards, k):
    """
    A big index is split across shards. Each shard returns its own top k (score,
    id); the coordinator merges them into the global top k. Return the global
    top-k ids and whether it equals searching all scores at once (it always does
    when every shard returns k).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shard_search, [
    (([{'a': 0.91, 'b': 0.55, 'c': 0.77}, {'d': 0.95, 'e': 0.4, 'f': 0.88}, {'g': 0.6, 'h': 0.93, 'i': 0.79}], 4), (['d', 'h', 'a', 'f'], True)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Concatenate everything and sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Each shard's top k, then heapq.merge.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Concatenate everything and sort.

```python
def shard_search(shards, k):
    everything = {i: s for sh in shards for i, s in sh.items()}
    top = sorted(everything, key=lambda i: (-everything[i], i))[:k]
    return (top, True)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Each shard's top k, then heapq.merge.

```python
def shard_search(shards, k):
    per_shard = [sorted(((-s, i) for i, s in sh.items()))[:k] for sh in shards]
    merged = [i for _, i in list(heapq.merge(*per_shard))[:k]]
    everything = {i: s for sh in shards for i, s in sh.items()}
    return (merged, merged == sorted(everything, key=lambda i: (-everything[i], i))[:k])
```

**Cost:** O(n log k + s·k log s) time, O(s · k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Concatenate everything and sort. That costs O(n log n) time and O(n) space."

"I can do better: Each shard's top k, then heapq.merge. That's O(n log k + s·k log s) time and O(s · k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/search-across-shards)

<a id="q13"></a>
### Q13. Hybrid Search With a Weighted Blend

🟡 Interview standard · *Keyword + vector scores*

Min-max normalise the BM25 scores and the vector scores separately to 0–1 (missing scores count as 0 after normalising; if all scores in a list are equal, they all become 1), then blend: `alpha × vector + (1 − alpha) × keyword`. Return the top-3 ids for each alpha (ties by id). alpha = 1 is pure vector search, 0 is pure keyword search.

**Examples**

- `bm25 = {'d1': 12.0, 'd2': 3.5, 'd4': 8.0, 'd6': 0.5}, vec = {'d1': 0.62, 'd2': 0.91, 'd3': 0.88, 'd5': 0.7}, alphas = [0.0, 0.5, 1.0]` → `{0.0: ['d1', 'd4', 'd2'], 0.5: ['d2', 'd1', 'd3'], 1.0: ['d2', 'd3', 'd5']}`

In [ ]:
def hybrid_blend(bm25, vec, alphas):
    """
    Min-max normalise the BM25 scores and the vector scores separately to 0–1
    (missing scores count as 0 after normalising; if all scores in a list are
    equal, they all become 1), then blend: alpha × vector + (1 − alpha) ×
    keyword. Return the top-3 ids for each alpha (ties by id). alpha = 1 is pure
    vector search, 0 is pure keyword search.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(hybrid_blend, [
    (({'d1': 12.0, 'd2': 3.5, 'd4': 8.0, 'd6': 0.5}, {'d1': 0.62, 'd2': 0.91, 'd3': 0.88, 'd5': 0.7}, [0.0, 0.5, 1.0]), {0.0: ['d1', 'd4', 'd2'], 0.5: ['d2', 'd1', 'd3'], 1.0: ['d2', 'd3', 'd5']}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Normalise with loops, blend per alpha.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A normaliser helper and sorted().

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Normalise with loops, blend per alpha.

```python
def hybrid_blend(bm25, vec, alphas):
    def norm(s):
        lo, hi = min(s.values()), max(s.values())
        return {k: (v - lo) / (hi - lo) if hi > lo else 1.0 for k, v in s.items()}
    kb, vb = norm(bm25), norm(vec)
    ids = sorted(set(bm25) | set(vec))
    out = {}
    for a in alphas:
        scores = {}
        for i in ids:
            scores[i] = a * vb.get(i, 0.0) + (1 - a) * kb.get(i, 0.0)
        out[a] = sorted(ids, key=lambda i: (-scores[i], i))[:3]
    return out
```

**Cost:** O(a · n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A normaliser helper and sorted().

```python
def hybrid_blend(bm25, vec, alphas):
    def norm(s):
        lo, hi = min(s.values()), max(s.values())
        return {k: (v - lo) / (hi - lo) if hi > lo else 1.0 for k, v in s.items()}
    kb, vb = norm(bm25), norm(vec)
    ids = bm25.keys() | vec.keys()
    score = lambda a, i: a * vb.get(i, 0.0) + (1 - a) * kb.get(i, 0.0)
    return {a: sorted(ids, key=lambda i: (-score(a, i), i))[:3] for a in alphas}
```

**Cost:** O(a · n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Normalise with loops, blend per alpha. That costs O(a · n log n) time and O(n) space."

"I can do better: A normaliser helper and sorted(). That's O(a · n log n) time and O(n) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `bm25 = {'a': 1.0, 'b': 2.0}, vec = {'a': 0.9, 'b': 0.2}, alphas = [0.5]`

1. norm() returns {'a': 0.0, 'b': 1.0}
2. norm() returns {'a': 1.0, 'b': 0.0}
3. hybrid\_blend() returns {0.5: ['a', 'b']}

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/hybrid-search-with-a-weighted-blend)

<a id="q14"></a>
### Q14. Evaluate Metadata Filters

🟡 Interview standard · *Chroma/Qdrant-style where clauses*

Vector stores filter on metadata with nested conditions: `{'field': value}`, `{'field': {'$gte': 2024}}` (also `$eq $ne $gt $lt $lte $in $nin`), combined with `$and` / `$or`. Return the ids that match each filter (use `matches`). A missing field never matches, except with `$ne`.

**Examples**

- `meta = {'c1': {'year': 2025, 'lang': 'en', 'team': 'billing'}, 'c2': {'year': 2023, 'lang': 'hi', 'team': 'billing'}, 'c3': {'year': 2024, 'lang': 'en'}, 'c4': {'year': 2026, 'lang': 'ta', 'team': 'support'}}, filters = [{'lang': 'en'}, {'$and': [{'year': {'$gte': 2024}}, {'lang': {'$in': ['en', 'hi']}}]}, {'$or': [{'team': 'support'}, {'year': {'$lt': 2024}}]}, {'team': {'$ne': 'billing'}}]` → `[['c1', 'c3'], ['c1', 'c3'], ['c2', 'c4'], ['c3', 'c4']]`

In [ ]:
def filter_ids(meta, filters):
    """
    Vector stores filter on metadata with nested conditions: {'field': value},
    {'field': {'$gte': 2024}} (also $eq $ne $gt $lt $lte $in $nin), combined
    with $and / $or. Return the ids that match each filter (use matches). A
    missing field never matches, except with $ne.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(filter_ids, [
    (({'c1': {'year': 2025, 'lang': 'en', 'team': 'billing'}, 'c2': {'year': 2023, 'lang': 'hi', 'team': 'billing'}, 'c3': {'year': 2024, 'lang': 'en'}, 'c4': {'year': 2026, 'lang': 'ta', 'team': 'support'}}, [{'lang': 'en'}, {'$and': [{'year': {'$gte': 2024}}, {'lang': {'$in': ['en', 'hi']}}]}, {'$or': [{'team': 'support'}, {'year': {'$lt': 2024}}]}, {'team': {'$ne': 'billing'}}]), [['c1', 'c3'], ['c1', 'c3'], ['c2', 'c4'], ['c3', 'c4']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop over documents for each filter.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension per filter.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop over documents for each filter.

```python
def matches(meta, cond):
    if "$and" in cond:
        return all(matches(meta, c) for c in cond["$and"])
    if "$or" in cond:
        return any(matches(meta, c) for c in cond["$or"])
    field, rule = next(iter(cond.items()))
    v = meta.get(field)
    if not isinstance(rule, dict):
        return v == rule
    op, arg = next(iter(rule.items()))
    if v is None:
        return op == "$ne"
    ops = {"$eq": lambda: v == arg, "$ne": lambda: v != arg, "$gt": lambda: v > arg, "$gte": lambda: v >= arg,
           "$lt": lambda: v < arg, "$lte": lambda: v <= arg, "$in": lambda: v in arg, "$nin": lambda: v not in arg}
    return ops[op]()


def filter_ids(meta, filters):
    out = []
    for f in filters:
        ids = []
        for i, m in meta.items():
            if matches(m, f):
                ids.append(i)
        out.append(ids)
    return out
```

**Cost:** O(f · n · c) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension per filter.

```python
def matches(meta, cond):
    if "$and" in cond:
        return all(matches(meta, c) for c in cond["$and"])
    if "$or" in cond:
        return any(matches(meta, c) for c in cond["$or"])
    field, rule = next(iter(cond.items()))
    v = meta.get(field)
    if not isinstance(rule, dict):
        return v == rule
    op, arg = next(iter(rule.items()))
    if v is None:
        return op == "$ne"
    ops = {"$eq": lambda: v == arg, "$ne": lambda: v != arg, "$gt": lambda: v > arg, "$gte": lambda: v >= arg,
           "$lt": lambda: v < arg, "$lte": lambda: v <= arg, "$in": lambda: v in arg, "$nin": lambda: v not in arg}
    return ops[op]()


def filter_ids(meta, filters):
    return [[i for i, m in meta.items() if matches(m, f)] for f in filters]
```

**Cost:** O(f · n · c) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop over documents for each filter. That costs O(f · n · c) time and O(n) space."

"I can do better: A comprehension per filter. That's O(f · n · c) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/evaluate-metadata-filters)

<a id="q15"></a>
### Q15. Vector Search in SQL

🟡 Interview standard · *pgvector-style queries with DuckDB*

Store chunks with an embedding column and search with SQL, as pgvector does in PostgreSQL: filter to the tenant, order by cosine similarity to the query, limit k. Return `(id, body, similarity rounded to 3)`. Keeping vectors next to your relational data makes tenant filters and joins easy.

**Examples**

- `chunks = [{'id': 1, 'tenant': 'acme', 'body': 'refund policy', 'emb': [0.9, 0.1, 0.1]}, {'id': 2, 'tenant': 'acme', 'body': 'shipping times', 'emb': [0.1, 0.9, 0.2]}, {'id': 3, 'tenant': 'globex', 'body': 'refund rules', 'emb': [0.85, 0.15, 0.05]}, {'id': 4, 'tenant': 'acme', 'body': 'return a parcel', 'emb': [0.7, 0.4, 0.1]}, {'id': 5, 'tenant': 'acme', 'body': 'invoice copy', 'emb': [0.3, 0.2, 0.9]}], tenant = 'acme', query = [1.0, 0.2, 0.0], k = 3` → `[(1, 'refund policy', 0.99), (4, 'return a parcel', 0.941), (5, 'invoice copy', 0.344)]`

In [ ]:
import duckdb

def sql_vector_search(chunks, tenant, query, k):
    """
    Store chunks with an embedding column and search with SQL, as pgvector does
    in PostgreSQL: filter to the tenant, order by cosine similarity to the
    query, limit k. Return (id, body, similarity rounded to 3). Keeping vectors
    next to your relational data makes tenant filters and joins easy.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sql_vector_search, [
    (([{'id': 1, 'tenant': 'acme', 'body': 'refund policy', 'emb': [0.9, 0.1, 0.1]}, {'id': 2, 'tenant': 'acme', 'body': 'shipping times', 'emb': [0.1, 0.9, 0.2]}, {'id': 3, 'tenant': 'globex', 'body': 'refund rules', 'emb': [0.85, 0.15, 0.05]}, {'id': 4, 'tenant': 'acme', 'body': 'return a parcel', 'emb': [0.7, 0.4, 0.1]}, {'id': 5, 'tenant': 'acme', 'body': 'invoice copy', 'emb': [0.3, 0.2, 0.9]}], 'acme', [1.0, 0.2, 0.0], 3), [(1, 'refund policy', 0.99), (4, 'return a parcel', 0.941), (5, 'invoice copy', 0.344)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Cosine similarity in a loop, then sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

DuckDB `list_cosine_similarity` in a query.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Cosine similarity in a loop, then sort.

```python
def sql_vector_search(chunks, tenant, query, k):
    out = []
    for c in chunks:
        if c["tenant"] != tenant:
            continue
        dot = sum(a * b for a, b in zip(c["emb"], query))
        sim = dot / (math.sqrt(sum(a * a for a in c["emb"])) * math.sqrt(sum(b * b for b in query)))
        out.append((c["id"], c["body"], round(sim, 3)))
    return sorted(out, key=lambda r: (-r[2], r[0]))[:k]
```

**Cost:** O(n · d) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** DuckDB `list_cosine_similarity` in a query.

```python
def sql_vector_search(chunks, tenant, query, k):
    con = duckdb.connect()
    con.execute("CREATE TABLE chunks (id BIGINT, tenant VARCHAR, body VARCHAR, emb DOUBLE[])")
    con.executemany("INSERT INTO chunks VALUES (?, ?, ?, ?)", [[c["id"], c["tenant"], c["body"], c["emb"]] for c in chunks])
    return con.execute("""
        SELECT id, body, ROUND(list_cosine_similarity(emb, CAST(? AS DOUBLE[])), 3) AS sim
        FROM chunks WHERE tenant = ? ORDER BY sim DESC, id LIMIT ?""", [query, tenant, k]).fetchall()
```

**Cost:** O(n · d) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Cosine similarity in a loop, then sort. That costs O(n · d) time and O(n) space."

"I can do better: DuckDB `list_cosine_similarity` in a query. That's O(n · d) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/vector-search-in-sql)

<a id="q16"></a>
### Q16. Plan an Embedding Model Migration

🟡 Interview standard · *Re-embed without downtime*

Switching embedding models means re-embedding every chunk into a **new** index while the old one serves traffic, then switching over. Given chunks, average tokens per chunk, the provider's rate limit (tokens per minute), the price per million tokens, and the share of capacity you can use, return `(hours to backfill rounded to 1, cost in $ rounded to 2, GB for both indexes during the switch rounded to 1)` with float32 vectors of the old and new dimensions.

**Examples**

- `chunks = 4000000, tokens_each = 350, tpm = 1000000, price = 0.02, share = 0.7, old_dim = 768, new_dim = 1024` → `(33.3, 28.0, 28.7)`

In [ ]:
def migration_plan(chunks, tokens_each, tpm, price, share, old_dim, new_dim):
    """
    Switching embedding models means re-embedding every chunk into a new index
    while the old one serves traffic, then switching over. Given chunks, average
    tokens per chunk, the provider's rate limit (tokens per minute), the price
    per million tokens, and the share of capacity you can use, return (hours to
    backfill rounded to 1, cost in $ rounded to 2, GB for both indexes during
    the switch rounded to 1) with float32 vectors of the old and new dimensions.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(migration_plan, [
    ((4000000, 350, 1000000, 0.02, 0.7, 768, 1024), (33.3, 28.0, 28.7)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step-by-step arithmetic.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, compact.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Step-by-step arithmetic.

```python
def migration_plan(chunks, tokens_each, tpm, price, share, old_dim, new_dim):
    tokens = chunks * tokens_each
    minutes = tokens / (tpm * share)
    cost = tokens / 1_000_000 * price
    gb = chunks * (old_dim + new_dim) * 4 / 1e9
    return (round(minutes / 60, 1), round(cost, 2), round(gb, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, compact.

```python
def migration_plan(chunks, tokens_each, tpm, price, share, old_dim, new_dim):
    t = chunks * tokens_each
    return (round(t / (tpm * share) / 60, 1), round(t * price / 1e6, 2), round(chunks * (old_dim + new_dim) * 4 / 1e9, 1))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step-by-step arithmetic. That costs O(1) time and O(1) space."

"I can do better: Same, compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/plan-an-embedding-model-migration)

<a id="q17"></a>
### Q17. Compose a Chain Like LangChain

🟡 Interview standard · *prompt | llm | parser*

LangChain's LCEL composes steps with `|`: each step's output feeds the next, and a dict of steps runs them in parallel on the same input. Given a pipeline of step names from `STEPS` (a step may also be a dict `{key: [step names]}` meaning 'run these sub-chains and collect a dict'), run it on the input. The brute force loops; the optimal builds a tiny `Runnable` class with `__or__`.

**Examples**

- `pipeline = ['prompt', 'llm', 'parse_json', 'pick_answer'], value = {'topic': 'refunds'}` → `'refunds is covered in our docs'`
- `pipeline = [{'short': ['prompt', 'llm', 'parse_json', 'pick_answer'], 'loud': ['prompt', 'llm', 'parse_json', 'pick_answer', 'upper']}], value = {'topic': 'shipping'}` → `{'short': 'shipping is covered in our docs', 'loud': 'SHIPPING IS COVERED IN OUR DOCS'}`

In [ ]:
import re

def fake_llm(prompt):
    topic = re.search(r"about (\w+)", prompt)
    return f'{{"answer": "{topic.group(1) if topic else "unknown"} is covered in our docs", "confidence": 0.8}}'

STEPS = {"prompt": lambda x: f"Answer briefly about {x['topic']}.", "llm": fake_llm,
         "parse_json": lambda s: json.loads(s), "pick_answer": lambda d: d["answer"], "upper": lambda s: s.upper()}

def run_chain(pipeline, value):
    """
    LangChain's LCEL composes steps with |: each step's output feeds the next,
    and a dict of steps runs them in parallel on the same input. Given a
    pipeline of step names from STEPS (a step may also be a dict {key: [step
    names]} meaning 'run these sub-chains and collect a dict'), run it on the
    input. The brute force loops; the optimal builds a tiny Runnable class with
    __or__.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(run_chain, [
    ((['prompt', 'llm', 'parse_json', 'pick_answer'], {'topic': 'refunds'}), 'refunds is covered in our docs'),
    (([{'short': ['prompt', 'llm', 'parse_json', 'pick_answer'], 'loud': ['prompt', 'llm', 'parse_json', 'pick_answer', 'upper']}], {'topic': 'shipping'}), {'short': 'shipping is covered in our docs', 'loud': 'SHIPPING IS COVERED IN OUR DOCS'}),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A loop applying each step.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Runnable objects composed with |.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A loop applying each step.

```python
def run_chain(pipeline, value):
    for step in pipeline:
        if isinstance(step, dict):
            value = {k: run_chain(sub, value) for k, sub in step.items()}
        else:
            value = STEPS[step](value)
    return value
```

**Cost:** O(s) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Runnable objects composed with |.

```python
def run_chain(pipeline, value):
    class Runnable:
        def __init__(self, fn):
            self.fn = fn
        def __or__(self, other):
            return Runnable(lambda x: other.fn(self.fn(x)))
        def invoke(self, x):
            return self.fn(x)
    def build(steps):
        def one(s):
            if isinstance(s, dict):
                parts = {k: build(v) for k, v in s.items()}
                return Runnable(lambda x: {k: r.invoke(x) for k, r in parts.items()})
            return Runnable(STEPS[s])
        chain = one(steps[0])
        for s in steps[1:]:
            chain = chain | one(s)
        return chain
    return build(pipeline).invoke(value)
```

**Cost:** O(s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A loop applying each step. That costs O(s) time and O(1) space."

"I can do better: Runnable objects composed with |. That's O(s) time and O(s) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `pipeline = ['prompt', 'upper'], value = {'topic': 'x'}`

1. Runnable() returns None
2. \_\_init\_\_() returns None
3. one() returns &lt;topic\_10\_vector\_db.run\_chain.&lt;locals...
4. \_\_init\_\_() returns None
5. one() returns &lt;topic\_10\_vector\_db.run\_chain.&lt;locals...
6. \_\_init\_\_() returns None
7. \_\_or\_\_() returns &lt;topic\_10\_vector\_db.run\_chain.&lt;locals...
8. build() returns &lt;topic\_10\_vector\_db.run\_chain.&lt;locals...
9. invoke() returns 'ANSWER BRIEFLY ABOUT X.'
10. run\_chain() returns 'ANSWER BRIEFLY ABOUT X.'

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/compose-a-chain-like-langchain)

<a id="q18"></a>
### Q18. Split Text Recursively

🟡 Interview standard · *RecursiveCharacterTextSplitter*

Split text into chunks of at most `size` characters, trying separators in order (`'\n\n'`, `'\n'`, `' '`, then single characters): split on the first separator that occurs, merge consecutive pieces (joined by that separator) while they fit, and recursively split any piece still too long with the next separators. Return the chunks.

**Examples**

- `text = 'Refunds\n\nWe refund unused items within 30 days.\nOpened items get store credit.\n\nShipping\n\nOrders ship in 2 days.', size = 40` → `['Refunds', 'We refund unused items within 30 days.', 'Opened items get store credit.', 'Shipping\n\nOrders ship in 2 days.']`

In [ ]:
def recursive_split(text, size, seps=('\n\n', '\n', ' ', '')):
    """
    Split text into chunks of at most size characters, trying separators in
    order ('\n\n', '\n', ' ', then single characters): split on the first
    separator that occurs, merge consecutive pieces (joined by that separator)
    while they fit, and recursively split any piece still too long with the next
    separators. Return the chunks.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(recursive_split, [
    (('Refunds\n\nWe refund unused items within 30 days.\nOpened items get store credit.\n\nShipping\n\nOrders ship in 2 days.', 40), ['Refunds', 'We refund unused items within 30 days.', 'Opened items get store credit.', 'Shipping\n\nOrders ship in 2 days.']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive splitting with explicit loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The same recursion with a merge helper.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Recursive splitting with explicit loops.

```python
def recursive_split(text, size, seps=("\n\n", "\n", " ", "")):
    if len(text) <= size:
        return [text] if text else []
    sep = next(s for s in seps if s == "" or s in text)
    rest = seps[seps.index(sep) + 1:]
    pieces = list(text) if sep == "" else text.split(sep)
    chunks, cur = [], ""
    for p in pieces:
        candidate = p if not cur else cur + sep + p
        if len(candidate) <= size:
            cur = candidate
        else:
            if cur:
                chunks.append(cur)
            if len(p) > size:
                chunks += recursive_split(p, size, rest)
                cur = ""
            else:
                cur = p
    if cur:
        chunks.append(cur)
    return chunks
```

**Cost:** O(n · s) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** The same recursion with a merge helper.

```python
def recursive_split(text, size, seps=("\n\n", "\n", " ", "")):
    if len(text) <= size:
        return [text] if text else []
    i = next(i for i, s in enumerate(seps) if s == "" or s in text)
    sep, rest = seps[i], seps[i + 1:]
    out, cur = [], ""
    for p in (list(text) if sep == "" else text.split(sep)):
        joined = f"{cur}{sep}{p}" if cur else p
        if len(joined) <= size:
            cur = joined
            continue
        out += [cur] if cur else []
        if len(p) > size:
            out += recursive_split(p, size, rest)
            cur = ""
        else:
            cur = p
    return out + ([cur] if cur else [])
```

**Cost:** O(n · s) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive splitting with explicit loops. That costs O(n · s) time and O(n) space."

"I can do better: The same recursion with a merge helper. That's O(n · s) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/split-text-recursively)

<a id="q19"></a>
### Q19. A Semantic Cache for LLM Answers

🟡 Interview standard · *Reuse answers to similar questions*

Cache LLM answers by question **embedding**: if a new question's cosine similarity to a cached question is at least `threshold`, reuse that answer; otherwise call the model and cache it. Return `(for each question: 'hit:*i*' with the cached question's index or 'miss', model calls)`. Too low a threshold returns wrong answers; too high saves nothing.

**Examples**

- `questions = [[0.9, 0.1, 0.0], [0.88, 0.12, 0.02], [0.1, 0.9, 0.1], [0.6, 0.6, 0.0], [0.12, 0.88, 0.1]], threshold = 0.97` → `(['miss', 'hit:0', 'miss', 'miss', 'hit:2'], 3)`
- `questions = [[0.9, 0.1, 0.0], [0.88, 0.12, 0.02], [0.1, 0.9, 0.1], [0.6, 0.6, 0.0], [0.12, 0.88, 0.1]], threshold = 0.75` → `(['miss', 'hit:0', 'miss', 'hit:0', 'hit:2'], 2)`

In [ ]:
def semantic_cache(questions, threshold):
    """
    Cache LLM answers by question embedding: if a new question's cosine
    similarity to a cached question is at least threshold, reuse that answer;
    otherwise call the model and cache it. Return (for each question: 'hit:i'
    with the cached question's index or 'miss', model calls). Too low a
    threshold returns wrong answers; too high saves nothing.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(semantic_cache, [
    (([[0.9, 0.1, 0.0], [0.88, 0.12, 0.02], [0.1, 0.9, 0.1], [0.6, 0.6, 0.0], [0.12, 0.88, 0.1]], 0.97), (['miss', 'hit:0', 'miss', 'miss', 'hit:2'], 3)),
    (([[0.9, 0.1, 0.0], [0.88, 0.12, 0.02], [0.1, 0.9, 0.1], [0.6, 0.6, 0.0], [0.12, 0.88, 0.1]], 0.75), (['miss', 'hit:0', 'miss', 'hit:0', 'hit:2'], 2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compare with every cached question in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A matrix of cached normalised vectors and one dot product.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compare with every cached question in a loop.

```python
def semantic_cache(questions, threshold):
    cache, out = [], []
    for q in questions:
        best, best_sim = None, -1.0
        for i, c in cache:
            sim = sum(a * b for a, b in zip(q, c)) / (math.sqrt(sum(a * a for a in q)) * math.sqrt(sum(b * b for b in c)))
            if sim > best_sim:
                best, best_sim = i, sim
        if best is not None and best_sim >= threshold:
            out.append(f"hit:{best}")
        else:
            out.append("miss")
            cache.append((len(out) - 1, q))
    return (out, sum(o == "miss" for o in out))
```

**Cost:** O(n² · d) time, O(n · d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A matrix of cached normalised vectors and one dot product.

```python
def normalise(v):
    v = np.asarray(v, dtype=float)
    return v / np.linalg.norm(v)


def semantic_cache(questions, threshold):
    keys, ids, out = [], [], []
    for n, q in enumerate(questions):
        v = normalise(q)
        if keys:
            sims = np.array(keys) @ v
            j = int(np.argmax(sims))
            if sims[j] >= threshold:
                out.append(f"hit:{ids[j]}")
                continue
        out.append("miss")
        keys.append(v)
        ids.append(n)
    return (out, out.count("miss"))
```

**Cost:** O(n² · d) time, O(n · d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compare with every cached question in a loop. That costs O(n² · d) time and O(n · d) space."

"I can do better: A matrix of cached normalised vectors and one dot product. That's O(n² · d) time and O(n · d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/a-semantic-cache-for-llm-answers)

<a id="q20"></a>
### Q20. Product Quantization

🔴 Stretch · *Compress vectors to a few bytes*

PQ splits each vector into `m` equal sub-vectors and replaces each with the index of its nearest codeword in that sub-space's codebook. Return the codes for each vector, the bytes per vector before (float32) and after (1 byte per code), and each vector's reconstruction error (Euclidean distance to the decoded vector, rounded to 3).

**Examples**

- `vectors = {'v1': [0.1, 0.9, 0.4, 0.4], 'v2': [0.8, 0.2, 0.9, 0.1], 'v3': [0.15, 0.85, 0.95, 0.05]}, codebooks = [[[0.1, 0.9], [0.8, 0.2]], [[0.4, 0.4], [0.9, 0.1], [0.0, 0.0]]]` → `({'v1': ([0, 0], 0.0), 'v2': ([1, 1], 0.0), 'v3': ([0, 1], 0.1)}, 16, 2)`

In [ ]:
def pq_encode(vectors, codebooks):
    """
    PQ splits each vector into m equal sub-vectors and replaces each with the
    index of its nearest codeword in that sub-space's codebook. Return the codes
    for each vector, the bytes per vector before (float32) and after (1 byte per
    code), and each vector's reconstruction error (Euclidean distance to the
    decoded vector, rounded to 3).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pq_encode, [
    (({'v1': [0.1, 0.9, 0.4, 0.4], 'v2': [0.8, 0.2, 0.9, 0.1], 'v3': [0.15, 0.85, 0.95, 0.05]}, [[[0.1, 0.9], [0.8, 0.2]], [[0.4, 0.4], [0.9, 0.1], [0.0, 0.0]]]), ({'v1': ([0, 0], 0.0), 'v2': ([1, 1], 0.0), 'v3': ([0, 1], 0.1)}, 16, 2)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loops over sub-vectors and codewords.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

NumPy reshape and argmin per sub-space.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loops over sub-vectors and codewords.

```python
def pq_encode(vectors, codebooks):
    m = len(codebooks)
    out = {}
    for name, v in vectors.items():
        s = len(v) // m
        codes, decoded = [], []
        for j in range(m):
            sub = v[j * s:(j + 1) * s]
            best = min(range(len(codebooks[j])), key=lambda c: sum((a - b) ** 2 for a, b in zip(sub, codebooks[j][c])))
            codes.append(best)
            decoded += codebooks[j][best]
        err = math.sqrt(sum((a - b) ** 2 for a, b in zip(v, decoded)))
        out[name] = (codes, round(err, 3))
    return (out, len(next(iter(vectors.values()))) * 4, m)
```

**Cost:** O(n · m · c · s) time, O(n · m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** NumPy reshape and argmin per sub-space.

```python
def pq_encode(vectors, codebooks):
    m = len(codebooks)
    out = {}
    for name, v in vectors.items():
        subs = np.array(v).reshape(m, -1)
        codes = [int(np.argmin(((np.array(codebooks[j]) - subs[j]) ** 2).sum(1))) for j in range(m)]
        decoded = np.concatenate([np.array(codebooks[j][c]) for j, c in enumerate(codes)])
        out[name] = (codes, round(float(np.linalg.norm(np.array(v) - decoded)), 3))
    return (out, len(next(iter(vectors.values()))) * 4, m)
```

**Cost:** O(n · m · c · s) time, O(n · m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loops over sub-vectors and codewords. That costs O(n · m · c · s) time and O(n · m) space."

"I can do better: NumPy reshape and argmin per sub-space. That's O(n · m · c · s) time and O(n · m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/ai-engineering/#/q/vector-db/product-quantization)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Why not just use NumPy for vector search?</b></summary>

Exact search is O(n·d) per query — fine for thousands of vectors, too slow for millions at high QPS. Databases add ANN indexes, filtering, persistence, updates, replication and access control.

</details>

<details>
<summary><b>2. What's the difference between pre- and post-filtering?</b></summary>

Post-filtering searches first and then drops non-matching results, which can leave fewer than k or none. Pre- or in-search filtering applies the condition during the search so you still get k matches.

</details>

<details>
<summary><b>3. How do you measure an ANN index?</b></summary>

Recall@k against exact search on a sample of real queries, plus latency percentiles and memory. Tune ef\_search or nprobe until recall is good enough for your RAG evaluation.

</details>

<details>
<summary><b>4. How do you change embedding models?</b></summary>

Re-embed everything into a new index (vectors from different models aren't comparable), dual-write new documents during the backfill, evaluate retrieval on both, then switch reads and delete the old index.

</details>

<details>
<summary><b>5. What is a semantic cache?</b></summary>

A cache keyed by query embeddings: a new question close enough to a previous one reuses its answer. It saves money and latency, but the threshold must be tuned — too loose returns wrong answers.

</details>

<details>
<summary><b>6. Should you use LangChain or LlamaIndex?</b></summary>

They're useful for prototypes and integrations. Know what each abstraction does underneath — chunking, retrieval, prompts, parsing — so you can debug it or replace it with plain code when you need control.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: What a vector database does
- [ ] I can explain: Inside ANN indexes
- [ ] I can explain: Vector search in practice
- [ ] I can explain: The same ideas in LangChain and LlamaIndex

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Plan an Embedding Model Migration
- [ ] [Q17](#q17) Compose a Chain Like LangChain
- [ ] [Q18](#q18) Split Text Recursively
- [ ] [Q19](#q19) A Semantic Cache for LLM Answers
- [ ] [Q20](#q20) Product Quantization

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.